# 3.4.2 PLE 渐进式专家抽取

> 阅读版与 Web 应用内容一致；实验数值来自本 Notebook 的已执行输出。

## Goal

如果共享专家仍让任务互相干扰，怎样显式保留任务专属知识，并逐层控制共享？

## Setup

本 Notebook 的默认真实数据是 **Census-Income KDD：MMoE/PLE 论文公开多任务实验的完整官方 train/test**。`smoke` 档读取仓库内可审计的确定性切片，`full` 档扩大到官方完整文件；两档都不制造交互、曝光、标签或行为序列。切片规则、源地址、哈希与许可记录在 `data/README.md` 及对应数据目录。

**主要资料：** [Tang et al., 2020, PLE](https://dl.acm.org/doi/10.1145/3383313.3412236)

In [ ]:
from pathlib import Path
import os, sys, json
import torch
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ARTIFACT_ROOT = Path(os.environ.get("RECSYS_ARTIFACT_ROOT", PROJECT_ROOT)).expanduser().resolve()
sys.path.insert(0, str(PROJECT_ROOT))
os.environ.setdefault("RECSYS_PROFILE", "full")
PROFILE = os.environ["RECSYS_PROFILE"]
CUDA_AVAILABLE = torch.cuda.is_available()
from recsys_lab.data import (load_movielens, movielens_provenance, load_amazon_2023,
                             amazon_provenance, load_kuairand, kuairand_provenance,
                             load_amazon_2018, amazon_2018_provenance,
                             load_movielens_1m, movielens_1m_provenance,
                             load_mind_amazon_books, mind_amazon_provenance,
                             load_census_income, census_income_provenance)
DATASET_KEY = "census-income"
if DATASET_KEY == "movielens":
    real_ratings, real_movies = load_movielens()
    real_interactions = real_ratings
    REAL_DATASET = movielens_provenance(real_ratings)
elif DATASET_KEY == "amazon-books":
    real_ratings = load_amazon_2018("Books") if PROFILE == "full" else load_amazon_2023()
    real_interactions, real_movies = real_ratings, None
    REAL_DATASET = amazon_2018_provenance(real_ratings) if PROFILE == "full" else amazon_provenance(real_ratings)
elif DATASET_KEY == "mind-amazon-books":
    real_ratings = load_mind_amazon_books() if PROFILE == "full" else load_amazon_2023()
    real_interactions, real_movies = real_ratings, None
    REAL_DATASET = mind_amazon_provenance(real_ratings) if PROFILE == "full" else amazon_provenance(real_ratings)
elif DATASET_KEY == "amazon-electronics":
    real_ratings = load_amazon_2018("Electronics") if PROFILE == "full" else load_amazon_2023()
    real_interactions, real_movies = real_ratings, None
    REAL_DATASET = amazon_2018_provenance(real_ratings) if PROFILE == "full" else amazon_provenance(real_ratings)
elif DATASET_KEY == "movielens-1m":
    real_ratings = load_movielens_1m() if PROFILE == "full" else load_amazon_2023()
    real_interactions, real_movies = real_ratings, None
    REAL_DATASET = movielens_1m_provenance(real_ratings) if PROFILE == "full" else amazon_provenance(real_ratings)
elif DATASET_KEY == "census-income" and PROFILE == "full":
    census_train_x, census_train_y, census_test_x, census_test_y = load_census_income()
    real_interactions, real_movies, real_ratings = census_train_x, None, census_train_x
    REAL_DATASET = census_income_provenance()
elif DATASET_KEY == "amazon-2023":
    real_ratings = load_amazon_2023()
    real_interactions, real_movies = real_ratings, None
    REAL_DATASET = amazon_provenance(real_ratings)
else:
    real_interactions, real_movies = load_kuairand()
    real_ratings = real_interactions
    REAL_DATASET = kuairand_provenance(real_interactions)
print({"profile": PROFILE, "project_root": str(PROJECT_ROOT), "artifact_root": str(ARTIFACT_ROOT), "real_dataset": REAL_DATASET,
       "cuda_available": CUDA_AVAILABLE,
       "cuda_device": torch.cuda.get_device_name(0) if CUDA_AVAILABLE else None})
assert REAL_DATASET["randomly_fabricated_rows"] == 0

## 学习地图

1. 从原始论文理解系统约束；
2. 用可手算数字读懂公式和形状；
3. 检查数据、切分与标签；
4. 使用工业框架模型类训练；
5. 分开验证训练、推理和测试；
6. 用实际输出讨论失败边界。

**本节问题：** 如果共享专家仍让任务互相干扰，怎样显式保留任务专属知识，并逐层控制共享？

**阅读约定：** 通用数学通过 3.0 基础课程链接回看；本页只详细推导论文引入或改造的数学。第一次阅读先追踪输入、输出和形状，再看梯度。

## Paper & Context

PLE（RecSys 2020，腾讯）先命名并量化跷跷板现象：在腾讯视频推荐的 VTR/VCR 复杂相关任务组上，当时 SOTA 多任务模型提升一个任务常牺牲另一个（Figure 3 中无基线落在双优象限）。它给每个任务独立专属专家、CGC 门控只读本任务与共享专家，再堆叠成多层渐进分离路由；代价是专家数、层数与动态损失权重等更多超参。

**来源：** [Tang et al., 2020, PLE](https://dl.acm.org/doi/10.1145/3383313.3412236)

### 原文实验设计与关键结论

原文工业数据：腾讯视频推荐 8 天日志采样，46.926M 用户、2.682M 视频、0.995B 样本，前 7 天训练。Table 1（复杂相关 VTR/VCR）：PLE VCR MSE 0.1307 最优、VTR AUC 0.6831，双任务 MTL gain 同时为正；MMoE 的 VCR gain 仅 +0.0001，ML-MMOE 升 VTR 损 VCR，即跷跷板。Table 3（4 周在线 A/B）：PLE 相对单任务总播放量 +4.17%、总观看时长 +3.57%（CGC +3.92%/+2.75%，MMoE +1.94%/+1.73%）；摘要另给相对 SOTA MTL +2.23%/+1.84% 的口径。Table 5：Census-income 两任务 0.9522/0.9945、Ali-CCP 0.6112/0.6097，均超单任务与 MMoE（MMoE 在 Ali-CCP CVR 上 gain −0.0302）；合成数据 PLE 平均 MTL gain 高 MMoE 87.2%。教程数值另行报告，不可相减。

请区分三层证据：论文中的离线实验、本 Notebook 验证的代码链路、生产系统尚需验证的在线收益。三者不能互相替代。

## Reproduction Contract

**正式数据：** Census-Income KDD  
**资源 ID：** `census-income-kdd`  
**切分：** official train file plus 1:1 validation/test split of the official test file  
**指标：** income AUC, marital-status AUC  
**与论文比较边界：** PLE reports Census-income offline experiments; Tencent online values remain proprietary

`full` 只有在运行输出证明数据、切分、候选集、模型配置与指标均对齐时，才可能进入论文数值比较；它不是把教学适配器自动变成论文复现的开关。`smoke` 只做张量、损失和推理链路回归。

## Model Structure & Formula Walkthrough

![Figure 4 · Customized Gate Control](/static/paper-figures/ple.webp)

> **论文原图节选** · Figure 4 · Customized Gate Control · PDF p.5。下图直接截取自原文，用于对照下方公式与代码。

### 关键模块

- **任务专属专家**：为每个任务保留一组其他任务不可见的专家，避免强制共享带来的负迁移。
- **共享专家**：额外一组专家对所有任务可见，专门学习可复用的公共表示。
- **逐层 CGC 路由**：每层每个任务的 gate 只能从自己的专家 + 共享专家中选；多层堆叠后公共与任务特有信息逐步分离。

### 结构：共享专家之外，再给每个任务专属空间

MMoE 的每个 gate 面对所有专家；CGC（Customized Gate Control）先改「谁能被选」。第 $l$ 层有三组专家：任务 A 的专属专家 $m_A$ 个、任务 B 的专属专家 $m_B$ 个、所有任务共用的共享专家 $m_s$ 个。$E_k^{(l)}$ 表示任务 $k$ 在第 $l$ 层的专属专家集合，$E_s^{(l)}$ 表示共享专家集合。

任务 $k$ 的 gate 只在自己的专属专家和共享专家之间分配权重：

$$w_k^{(l)}(x)=\mathrm{softmax}(W_{g,k}^{(l)}x)\in\mathbb R^{m_k+m_s},\qquad z_k^{(l)}=\sum_{e\in E_k^{(l)}\cup E_s^{(l)}}w_{k,e}^{(l)}(x)\,f_e^{(l)}(x).$$

连通性一句话讲清：任务 A 的 gate 读「A 专属专家 + 共享专家」，任务 B 的 gate 读「B 专属专家 + 共享专家」；其它任务的专属专家被结构性排除，权重恒为 0，梯度不再互相污染。论文把允许被选的专家输出拼成选择矩阵 $S_k(x)$，门控输出写作 $g_k(x)=w_k(x)\,S_k(x)$（公式 2-4），再过任务塔 $y_k=t_k(g_k(x))$（公式 5）。

手算一组：$m_k=1, m_s=1$，任务 A 的 gate logits 为 $(1,0)$，softmax 权重为 $(e/(e+1),\ 1/(e+1))\approx(0.731,0.269)$。若 A 专属专家输出 $(2,0)$、共享专家输出 $(0,2)$，融合结果为 $0.731\times(2,0)+0.269\times(0,2)=(1.462,\ 0.538)$。若这是 MMoE，B 的专属专家输出（比如 $(0,-2)$）也会进入 A 的候选集，权重一旦非零就成为干扰；CGC 从结构上让它恒为 0。

PLE 把 CGC 堆成多层，称为 progressive extraction（渐进抽取）：第 $j$ 层任务 $k$ 的门控以第 $j-1$ 层的融合结果为输入，即 $g_{k,j}(x)=w_{k,j}(g_{k,j-1}(x))\,S_{k,j}(x)$；共享门控的选择矩阵包含本层全部专家，产出公共表示送给下一层。越往高层，共享知识与任务专属知识的边界越清晰——论文用化学提纯类比：低层先充分混合萃取，高层再逐步分离。最终仍是每任务 tower、Sigmoid 与 $L=\sum_k\lambda_kL_k$，PLE 论文另给 $\lambda_k$ 配动态更新 $\omega_k^{(t)}=\omega_{k,0}\,\gamma_k^t$，并用样本掩码处理「先点击才可能有评论」造成的样本空间错位。与 MMoE 的关键差异不在损失，而在「哪些专家允许被哪个 gate 读取」。

### 公式到代码

`run_ple` 保持与 MMoE 相同数据和目标，仅替换为 Torch-RecHub PLE；这样结果差异才主要来自 CGC 结构而非样本口径。

阅读源码时按“张量形状 → 前向计算 → score → loss → metric”五步追踪，不需要一次读完整个工程文件。

## Math by Hand

### 通用先修（先回看 3.0 基础课程）

- [多任务标签](/notebooks/3_0_1_data_ml_basics#observation-label)
- [Softmax 与加权和](/notebooks/3_0_5_information_theory#softmax-temperature)
- [多任务梯度冲突](/notebooks/3_0_6_optimization#gradient-conflict)
- [正则化与泛化](/notebooks/3_0_6_optimization#regularization)

### 本论文新增数学（本节详细推导）

CGC 的可见专家约束、多层渐进分离，以及样本空间掩码和动态任务权重。

CGC 的关键是「谁能被选」：任务 $k$ 的 gate 只在自己的 $m_k$ 个专属专家和 $m_s$ 个共享专家之间分配 softmax 权重，其它任务的专属专家被结构性排除，权重恒为 0。手算一组：$m_k=1, m_s=1$，gate logits $(1,0)$ 得权重 $(e/(e+1), 1/(e+1))\approx(0.731,0.269)$；若专属专家输出 $(2,0)$、共享专家输出 $(0,2)$，融合结果为 $0.731\times(2,0)+0.269\times(0,2)=(1.462,0.538)$。直觉像公共课与专业课：都听公共课，但不被迫共享全部专业细节。堆叠多层即 progressive extraction：低层充分混合，高层逐步分离。

下面用 NumPy/Matplotlib 验证直觉。二维图只是教学投影，工业 embedding 虽有更多维，计算规则相同。

In [ ]:
import numpy as np, matplotlib.pyplot as plt
matrix=np.array([[1,1,0],[1,0,1]],dtype=float); fig,ax=plt.subplots(figsize=(5.5,2.8)); image=ax.imshow(matrix,cmap='YlGn')
ax.set_xticks(range(3),['shared','click-only','convert-only']); ax.set_yticks(range(2),['click gate','convert gate'])
for i in range(2):
    for j in range(3): ax.text(j,i,int(matrix[i,j]),ha='center',va='center')
ax.set_title('PLE: shared plus task-specific experts'); plt.colorbar(image,ax=ax,ticks=[0,1]); plt.show()

## Data

### 权威 full 协议（效果验收目标）

**正式数据：** Census-Income KDD  
**资源 ID：** `census-income-kdd`  
**切分：** official train file plus 1:1 validation/test split of the official test file  
**指标：** income AUC, marital-status AUC  
**与论文比较边界：** PLE reports Census-income offline experiments; Tencent online values remain proprietary

### smoke 教学适配器（默认 runner 实际读取）

与 MMoE 使用同一 KuaiRand-Pure 真实曝光教学子集和 `is_click`/`long_view` 标签，使 smoke 横向差异主要来自路由结构。它不冒充 full 的 Census-Income 官方切分。

下方运行结果打印的 provenance 才是本次执行事实；若资源、统计或切分与 full 协议不一致，就必须标记为不可比较。

**防泄漏清单：**按时间切分；item 映射只表达已知目录，不读取测试标签；低评分或未点击负反馈均来自数据中的已观察行；序列只看预测时刻以前；测试集只在最后评价。CPU 档使用真实数据的确定性子集，**不是统一 benchmark 成绩**。

## Model & Framework

实际使用 torch_rechub.models.multi_task.PLE，执行两层 CGC、共享/专属专家与任务塔；full profile 映射 TorchEasyRec PLE。

smoke 档强调模型类、张量契约和指标链路真实可运行；full 档应替换原始数据、分布式配置、索引/服务和资源预算，而不是只增加 epoch。

In [ ]:
import inspect
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from importlib import import_module
from recsys_lab.runtime import save_records

# 算法实现就在当前章节目录，不再通过公共模块隐藏。
chapter_train = import_module("chapter_code.3_4_2_ple.train")
run_ple = chapter_train.run_ple

print("实际执行函数源码（包含数据、训练、推理和测试）：")
print(inspect.getsource(run_ple))

## Train & Inference

下一格固定 seed、构造数据、实例化模型、训练并进入推理路径。生成式章节在 CUDA 上执行完整评测；CPU 环境只验证缩小后的基本张量与约束链路。

In [ ]:
result = run_ple()
print({'framework': result['framework'], 'dataset': result.get('dataset', {}),
       'device': result.get('device'), 'validation_mode': result.get('validation_mode')})
print('inference contract:', '共享/专属专家逐层计算，各任务塔输出；服务关注参数量、专家并行和输出版本兼容。')
assert np.isfinite(result['loss_curve']).all()
print('loss:', round(result['loss_curve'][0],4), '→', round(result['loss_curve'][-1],4))

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(10.5,3.5))
axes[0].plot(result['loss_curve'],color='#4f8f00',lw=2); axes[0].set(title='Training loss',xlabel='epoch',ylabel='loss'); axes[0].grid(alpha=.2)
metrics={'click_auc': result['click_auc'], 'long_view_auc': result['long_view_auc']}
axes[1].bar(range(len(metrics)),list(metrics.values()),color=['#7ca832','#e1874b','#6d88a4'][:len(metrics)])
axes[1].set_xticks(range(len(metrics)),list(metrics),rotation=18); axes[1].set(title='Executed test metrics',ylim=(0,max(1.0,max(metrics.values())*1.15)))
for index,value in enumerate(metrics.values()): axes[1].text(index,value,f'{value:.3f}',ha='center',va='bottom')
plt.tight_layout(); plt.show(); display(pd.Series(metrics,name='value').to_frame())

In [ ]:
# 论文数字只能在数据、切分、候选和指标全部同口径时相减。
paper_protocol = json.loads('{"dataset": "Census-Income KDD", "resource": "census-income-kdd", "split": "official train file plus 1:1 validation/test split of the official test file", "metrics": ["income AUC", "marital-status AUC"], "paper_comparison": "PLE reports Census-income offline experiments; Tencent online values remain proprietary"}')
paper_targets = paper_protocol.get('paper_targets', {})
metric_key = {'HitRate@10':'paper_protocol_hr@10', 'NDCG@10':'paper_protocol_ndcg@10',
              'AUC':'auc', 'LogLoss':'logloss'}
dataset_name = result.get('dataset', {}).get('dataset', '')
dataset_aligned = paper_protocol.get('dataset', '').split(',')[0].casefold() in dataset_name.casefold()
comparison_eligible = PROFILE == 'full' and dataset_aligned
rows=[]
for paper_metric,target in paper_targets.items():
    result_key=metric_key.get(paper_metric)
    value=result.get(result_key) if result_key else None
    rows.append({'metric':paper_metric,'tutorial':value,'paper':target,
                 'absolute_gap':None if value is None or not comparison_eligible else float(value)-float(target),
                 'comparable':comparison_eligible and value is not None})
if rows:
    display(pd.DataFrame(rows))
    if not comparison_eligible:
        print('NOT COMPARABLE：当前运行的数据/协议与论文不完全一致，不计算复现差值。')
else:
    print('论文没有可公开、可同口径复现的绝对目标；本节只报告结构与公开协议验证。')

## Test & Results Discussion

In [ ]:
display(Markdown(f'''### 本次已执行结果

- 主指标 click_auc = **{result['click_auc']:.4f}**。
- 辅助指标 long_view_auc = **{result['long_view_auc']:.4f}**。
- 本节没有把不同任务的数值伪装成 baseline；章节总结只做同口径比较。
- 训练损失从 **{result['loss_curve'][0]:.4f}** 降到 **{result['loss_curve'][-1]:.4f}**。损失下降只说明优化工作，不等于泛化或业务收益。
- **结果解释：** 简单或小数据上 PLE 可能不如 MMoE；复杂结构只有在负迁移真实存在且数据充足时才可能回本。

### 工业边界

共享/专属专家逐层计算，各任务塔输出；服务关注参数量、专家并行和输出版本兼容。

上线前还需验证延迟、吞吐、内存/显存、数据新鲜度、校准、回滚和线上 A/B。
'''))

In [ ]:
record={
    'algorithm': 'PLE 渐进式专家抽取',
    'primary_metric': 'click_auc', 'primary_value': float(result['click_auc']),
    'secondary_metric': 'long_view_auc', 'secondary_value': float(result['long_view_auc']),
    'baseline_metric': None,
    'baseline_value': float(result[None]) if False else None,
    'framework': result['framework'], 'source_notebook': '3_4_2_ple',
    'validation_mode': result.get('validation_mode', 'standard'),
    'dataset': result['dataset']['dataset'],
    'randomly_fabricated_rows': int(result['dataset']['randomly_fabricated_rows'])
}
path=save_records('chapter_3_4','3_4_2_ple',[record]); print('saved:',path.relative_to(ARTIFACT_ROOT))

## Checks

自动断言用于防止数据、训练和指标链路静默失效，不是效果证明。

In [ ]:
assert result['loss_curve'][-1] < result['loss_curve'][0]
assert 0 <= float(result['click_auc']) <= 1
assert np.isfinite(float(result['long_view_auc']))
print('PASS：数据、训练、推理、测试和结果产物均已验证。')

## Next Steps

1. 换成对应公开数据的完整时间切分；2. 增加强 baseline 与消融；3. 记录效果、延迟和成本；4. 映射到 TorchEasyRec/官方 full profile；5. 只在相同候选与数据口径下比较。